# LRTIA - Targeted Coreference Test

**The hypothesis:** Pronouns and references depend on long-range context.

**The test:** 
1. Create sentences where a PRONOUN refers to an ANTECEDENT introduced earlier
2. Measure prediction of the pronoun with vs without the antecedent
3. In intact text: antecedent helps → big effect when masked
4. In shuffled text: antecedent is random → small effect when masked

This MUST show a difference if the model does coreference at all.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

## Create Controlled Coreference Examples

Structure: `[FILLER] [ANTECEDENT sentence] [FILLER] [PRONOUN sentence]`

The pronoun's prediction should depend on knowing the antecedent.

In [ ]:
# Create examples with clear antecedent-pronoun relationships
# Format: (antecedent_sentence, pronoun_sentence, pronoun_token)

COREF_EXAMPLES = [
    # Male antecedents -> "He"
    ("John walked into the room and sat down at his desk.",
     "He opened his laptop and started typing.", "He"),
    ("The professor entered the lecture hall carrying a stack of papers.",
     "He placed them on the podium and cleared his throat.", "He"),
    ("Michael had been waiting for this moment his entire career.",
     "He finally received the promotion he deserved.", "He"),
    ("The old man sat alone on the park bench watching the pigeons.",
     "He threw breadcrumbs to them one by one.", "He"),
    ("David finished his coffee and checked his watch nervously.",
     "He was going to be late for his interview.", "He"),
    
    # Female antecedents -> "She"
    ("Sarah noticed something strange in the garden this morning.",
     "She walked closer to investigate the unusual sound.", "She"),
    ("The doctor reviewed the test results carefully before speaking.",
     "She had good news for the patient waiting outside.", "She"),
    ("Maria had always dreamed of becoming a professional dancer.",
     "She practiced for hours every single day.", "She"),
    ("The young woman stood at the edge of the cliff watching the sunset.",
     "She felt at peace for the first time in months.", "She"),
    ("Emily opened the letter with trembling hands.",
     "She could not believe what she was reading.", "She"),
    
    # Plural antecedents -> "They"
    ("The children gathered around the birthday cake excitedly.",
     "They sang happy birthday as loud as they could.", "They"),
    ("The researchers published their findings in a major journal.",
     "They had worked on this project for over five years.", "They"),
    ("The birds landed on the telephone wire one after another.",
     "They sat there quietly watching the street below.", "They"),
    
    # Object antecedents -> "It"
    ("The old clock on the wall suddenly stopped ticking.",
     "It had been running continuously for over fifty years.", "It"),
    ("The computer crashed right in the middle of my presentation.",
     "It had never done anything like that before.", "It"),
    ("The storm approached the coastal town with frightening speed.",
     "It brought winds of over one hundred miles per hour.", "It"),
]

# Neutral filler sentences (no pronouns that could confuse things)
FILLERS = [
    "The weather was pleasant that afternoon.",
    "Several cars passed by on the street.",
    "The clock on the wall showed half past three.",
    "A gentle breeze rustled the leaves outside.",
    "The coffee shop across the street was busy.",
    "Mountains could be seen in the distance.",
    "The newspaper lay unopened on the table.",
    "Clouds drifted slowly across the blue sky.",
]

print(f"Created {len(COREF_EXAMPLES)} coreference examples")

In [ ]:
def create_test_passages(examples, fillers, n_filler_before=3, n_filler_between=2):
    """
    Create intact and shuffled versions of passages.
    
    Intact: [fillers] [antecedent] [fillers] [pronoun sentence]
    Shuffled: [fillers] [RANDOM sentence] [fillers] [pronoun sentence]
    
    In shuffled, the antecedent is replaced with a random sentence,
    breaking the coreference chain.
    """
    passages = []
    rng = random.Random(42)
    
    for i, (antecedent, pronoun_sent, pronoun) in enumerate(examples):
        # Select fillers
        selected_fillers = rng.sample(fillers, n_filler_before + n_filler_between)
        fillers_before = selected_fillers[:n_filler_before]
        fillers_between = selected_fillers[n_filler_before:]
        
        # INTACT: proper coreference
        intact_sentences = fillers_before + [antecedent] + fillers_between + [pronoun_sent]
        intact_text = " ".join(intact_sentences)
        
        passages.append({
            "id": f"intact_{i}",
            "population": "intact",
            "text": intact_text,
            "antecedent": antecedent,
            "pronoun_sent": pronoun_sent,
            "pronoun": pronoun,
        })
        
        # SHUFFLED: break coreference by using wrong antecedent
        # Pick a random antecedent from a DIFFERENT example (different gender/number)
        other_examples = [e for j, e in enumerate(examples) if j != i and e[2] != pronoun]
        if other_examples:
            wrong_antecedent = rng.choice(other_examples)[0]
        else:
            wrong_antecedent = rng.choice(fillers)
        
        shuffled_sentences = fillers_before + [wrong_antecedent] + fillers_between + [pronoun_sent]
        shuffled_text = " ".join(shuffled_sentences)
        
        passages.append({
            "id": f"shuffled_{i}",
            "population": "shuffled",
            "text": shuffled_text,
            "antecedent": wrong_antecedent,  # Wrong antecedent!
            "pronoun_sent": pronoun_sent,
            "pronoun": pronoun,
        })
    
    return passages

passages = create_test_passages(COREF_EXAMPLES, FILLERS)
print(f"Created {len(passages)} test passages")
print(f"  - {sum(1 for p in passages if p['population']=='intact')} intact")
print(f"  - {sum(1 for p in passages if p['population']=='shuffled')} shuffled")

In [ ]:
# Show example
print("EXAMPLE INTACT:")
print(passages[0]['text'])
print(f"\nPronoun to predict: '{passages[0]['pronoun']}'")
print("\n" + "="*60)
print("\nEXAMPLE SHUFFLED (wrong antecedent):")
print(passages[1]['text'])
print(f"\nPronoun to predict: '{passages[1]['pronoun']}' (but antecedent doesn't match!)")

## Measure Pronoun Prediction

For each passage, measure how well the model predicts the pronoun.
- Intact: model should predict pronoun confidently (correct antecedent)
- Shuffled: model should be less confident (wrong antecedent gender/number)

In [ ]:
@torch.no_grad()
def get_pronoun_probability(text, pronoun):
    """
    Get the probability the model assigns to the pronoun at its position.
    Returns (probability, rank, top_predictions)
    """
    # Find where the pronoun sentence starts
    tokens = tokenizer.encode(text, return_tensors="pt").to(model.device)
    
    # Get the token ID for the pronoun
    pronoun_with_space = " " + pronoun  # Pronouns usually have space before
    pronoun_ids = tokenizer.encode(pronoun_with_space, add_special_tokens=False)
    if len(pronoun_ids) > 0:
        pronoun_token_id = pronoun_ids[0]
    else:
        pronoun_token_id = tokenizer.encode(pronoun, add_special_tokens=False)[0]
    
    # Find position of pronoun in the text
    token_list = tokens[0].tolist()
    pronoun_pos = None
    for i, tid in enumerate(token_list):
        if tid == pronoun_token_id:
            # Check if this is in the latter part of the text (the pronoun sentence)
            if i > len(token_list) // 2:  # Pronoun should be in second half
                pronoun_pos = i
                break
    
    if pronoun_pos is None:
        # Try without space
        pronoun_ids_no_space = tokenizer.encode(pronoun, add_special_tokens=False)
        if pronoun_ids_no_space:
            pronoun_token_id = pronoun_ids_no_space[0]
            for i, tid in enumerate(token_list):
                if tid == pronoun_token_id and i > len(token_list) // 2:
                    pronoun_pos = i
                    break
    
    if pronoun_pos is None:
        return None, None, None
    
    # Get model predictions at position before pronoun
    outputs = model(tokens)
    logits = outputs.logits[0, pronoun_pos - 1]  # Predict at position before pronoun
    probs = torch.softmax(logits, dim=-1)
    
    # Get probability of correct pronoun
    prob = probs[pronoun_token_id].item()
    
    # Get rank
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    rank = (sorted_ids == pronoun_token_id).nonzero(as_tuple=True)[0].item() + 1
    
    # Get top 5 predictions
    top5_ids = sorted_ids[:5].tolist()
    top5_probs = sorted_probs[:5].tolist()
    top5 = [(tokenizer.decode([tid]), p) for tid, p in zip(top5_ids, top5_probs)]
    
    return prob, rank, top5

In [ ]:
# Test on all passages
results = []

for p in tqdm(passages, desc="Testing"):
    prob, rank, top5 = get_pronoun_probability(p['text'], p['pronoun'])
    
    if prob is not None:
        results.append({
            "id": p['id'],
            "population": p['population'],
            "pronoun": p['pronoun'],
            "probability": prob,
            "rank": rank,
            "log_prob": np.log(prob) if prob > 0 else -100,
            "top5": top5,
        })

df = pd.DataFrame(results)
print(f"\nCollected {len(df)} measurements")

In [ ]:
print("=" * 70)
print("RESULTS: Pronoun Prediction with Correct vs Wrong Antecedent")
print("=" * 70)

print("\nMean probability of pronoun:")
print(df.groupby('population')['probability'].mean())

print("\nMean rank of pronoun:")
print(df.groupby('population')['rank'].mean())

print("\nMean log probability:")
print(df.groupby('population')['log_prob'].mean())

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Probability
ax = axes[0]
intact_probs = df[df['population']=='intact']['probability']
shuffled_probs = df[df['population']=='shuffled']['probability']
ax.bar(['Intact\n(correct antecedent)', 'Shuffled\n(wrong antecedent)'], 
       [intact_probs.mean(), shuffled_probs.mean()],
       color=['#2ecc71', '#e74c3c'],
       yerr=[intact_probs.std()/np.sqrt(len(intact_probs)), shuffled_probs.std()/np.sqrt(len(shuffled_probs))],
       capsize=5)
ax.set_ylabel('P(pronoun)')
ax.set_title('Pronoun Probability')

# Rank
ax = axes[1]
intact_ranks = df[df['population']=='intact']['rank']
shuffled_ranks = df[df['population']=='shuffled']['rank']
ax.bar(['Intact', 'Shuffled'], 
       [intact_ranks.mean(), shuffled_ranks.mean()],
       color=['#2ecc71', '#e74c3c'],
       yerr=[intact_ranks.std()/np.sqrt(len(intact_ranks)), shuffled_ranks.std()/np.sqrt(len(shuffled_ranks))],
       capsize=5)
ax.set_ylabel('Rank (lower is better)')
ax.set_title('Pronoun Rank')

# By pronoun type
ax = axes[2]
for pop, color in [('intact', '#2ecc71'), ('shuffled', '#e74c3c')]:
    pop_df = df[df['population']==pop]
    by_pronoun = pop_df.groupby('pronoun')['probability'].mean()
    x = np.arange(len(by_pronoun))
    width = 0.35
    offset = -width/2 if pop == 'intact' else width/2
    ax.bar(x + offset, by_pronoun.values, width, label=pop, color=color)
ax.set_xticks(range(len(by_pronoun)))
ax.set_xticklabels(by_pronoun.index)
ax.set_ylabel('P(pronoun)')
ax.set_title('By Pronoun Type')
ax.legend()

plt.tight_layout()
plt.savefig('pronoun_test.png', dpi=150)
plt.show()

In [ ]:
# Statistical test
from scipy import stats

intact_probs = df[df['population']=='intact']['probability']
shuffled_probs = df[df['population']=='shuffled']['probability']

t, p = stats.ttest_ind(intact_probs, shuffled_probs)
print(f"\nT-test: t={t:.3f}, p={p:.4f}")
print(f"Effect size (Cohen's d): {(intact_probs.mean() - shuffled_probs.mean()) / np.sqrt((intact_probs.std()**2 + shuffled_probs.std()**2)/2):.3f}")

if p < 0.05:
    print("\n*** SIGNIFICANT DIFFERENCE DETECTED ***")
    if intact_probs.mean() > shuffled_probs.mean():
        print("Intact (correct antecedent) → HIGHER pronoun probability")
        print("This confirms the model uses long-range context for coreference!")
else:
    print("\nNo significant difference detected.")

In [ ]:
# Show detailed examples
print("\n" + "=" * 70)
print("DETAILED EXAMPLES")
print("=" * 70)

for i in range(min(5, len(df)//2)):
    intact_row = df[(df['population']=='intact')].iloc[i]
    shuffled_row = df[(df['population']=='shuffled')].iloc[i]
    
    print(f"\n--- Example {i+1}: Pronoun '{intact_row['pronoun']}' ---")
    print(f"Intact:   P={intact_row['probability']:.4f}, Rank={intact_row['rank']}")
    print(f"Shuffled: P={shuffled_row['probability']:.4f}, Rank={shuffled_row['rank']}")
    print(f"Top predictions (intact):   {intact_row['top5'][:3]}")
    print(f"Top predictions (shuffled): {shuffled_row['top5'][:3]}")

In [ ]:
# Save
df.to_csv('pronoun_test_results.csv', index=False)
try:
    from google.colab import files
    files.download('pronoun_test_results.csv')
    files.download('pronoun_test.png')
except:
    pass